In [1]:
import pytest
import ipytest
from nodemanager import *

In [2]:
ipytest.autoconfig()

In [3]:
%%ipytest
@pytest.fixture
def mock_node_builder():
    """Mocks a NodeBuilder with a complete TIA dictionary and hybrid setup."""
    class MockNB:
        def __init__(self):
            # Physical locations
            self.positions = np.array([[0,0,0], [1,1,1], [2,2,2]])
            self.no_sensors = 3
            self.BW = 1e6
            
            # Reception properties
            self.rx_area = 1e-4
            self.rx_type = np.array([0, 0, 1]) # PD, PD, PV (Solar Panel)
            self.FOV = np.pi/2
            self.nR = np.array([[0,0,1], [0,0,1], [0,0,1]])
            
            # Transmission properties
            self.nT = np.array([[0,0,1], [0,0,1]]) # 2 optical emitters
            self.m = 1
            self.IR_tx_power = 0.5
            self.RF_tx_power = 0.1
            self.uplink_type = np.array([0, 1, 0]) # Node 0: Optical, 1: RF, 2: Optical
            self.no_optical_uplinks = 2
            self.no_RF_uplinks = 1
            
            # Filtering
            self.VLC_pass_filter = np.array([True, False, False])
            
            # TIA dictionary with all REQUIRED keys for the TIA class
            self.tia = {
                'RF': 10e3,           # Feedback resistance
                'Vn': 5e-9,           # Voltage noise
                'In': 10e-15,         # Current noise
                'fncV': 100,          # 1/f corner (Voltage)
                'fncI': 100,          # 1/f corner (Current)
                'temperature': 298    # Kelvin
            }
    return MockNB()

def test_sn_manager_initialization(mock_node_builder):
    """Verify element counts and flags."""
    sn = SNManager(mock_node_builder)
    
    assert sn.no_sensors == 3
    assert sn.ORx_elements.N == 3
    assert sn.OTx_elements.N == 2
    assert sn.RFTx_elements.N == 1
    assert sn.tia.RF == 10e3

def test_masking_logic(mock_node_builder):
    """Verify that filter and PV masks are generated correctly."""
    sn = SNManager(mock_node_builder)
    
    # Check PD vs PV segregation
    assert sn.mask_pv[2] == True
    assert sn.mask_pv[0] == False
    
    # Check Filter segregation
    assert sn.mask_VLC_filter[0] == True
    assert sn.mask_pd_no_filter[1] == True

def test_responsivity_assignment(mock_node_builder, monkeypatch):
    """Verify the spectral logic for PD and PV nodes."""
    
    def mock_get_resp(name):
        mapping = {
            "WLED2PDwF": 0.45,
            "WLED2PV": 0.1,
            "SUN2PV": 0.05,
            "WLED2PD": 0.5,
            "SUN2PDwFv": 0.01,
            "SUN2PD": 0.02
        }
        return mapping.get(name, 0.0)

    monkeypatch.setattr(SpectralPhysics, "get_responsivity_by_name", mock_get_resp)
    
    sn = SNManager(mock_node_builder)
    
    # PD with Filter
    assert sn.c_d[0] == 0.45
    # PD without Filter
    assert sn.c_d[1] == 0.5
    # PV (Solar Panel) logic: WLED2PV / SUN2PV = 0.1 / 0.05 = 2.0
    assert sn.c_d[2] == pytest.approx(2.0)

...                                                                                          [100%]
3 passed in 0.02s


In [4]:
%%ipytest

@pytest.fixture
def mock_node_builder_master():
    """Mocks a NodeBuilder with a dual-Master setup."""
    class MockNB:
        def __init__(self):
            # 2 Masters
            self.positions = np.array([[1,1,3], [4,4,3]])
            self.no_masters = 2
            self.BW = 20e6
            self.sensitivity = -90
            
            # Master Hardware (Downlink Tx, Uplink Rx)
            self.rx_area = 1e-4
            self.FOV = np.pi/3
            self.nR = np.array([[0,0,-1], [0,0,-1]]) # Looking down
            self.nT = np.array([[0,0,-1], [0,0,-1]]) # Pointing down
            self.m = 1
            self.tx_power = 10.0 # High power for base station
            
            # Filtering: One with IR-pass filter, one without
            self.IR_pass_filter = np.array([True, False])
            
            self.tia = {
                'RF': 5e3, 
                'Vn': 2e-9, 
                'In': 5e-15, 
                'fncV': 50, 
                'fncI': 50, 
                'temperature': 300
            }
    return MockNB()

def test_mn_manager_initialization(mock_node_builder_master):
    """Verify Master node counts, transceiver geometry, and sensitivity."""
    mn = MNManager(mock_node_builder_master)
    
    assert mn.no_masters == 2
    # Check that both Tx and Rx elements exist for both nodes
    assert mn.ORx_elements.N == 2
    assert mn.OTx_elements.N == 2
    
    # Sensitivity should be broadcasted to (2, 1)
    assert mn.sensitivity.shape == (2, 1)
    assert mn.sensitivity[0] == -90

def test_masking_ir_logic(mock_node_builder_master):
    """Verify that IR filter masks are correctly generated."""
    mn = MNManager(mock_node_builder_master)
    
    # Master 0 has IR filter, Master 1 does not
    assert mn.mask_IR_filter[0] == True
    assert mn.mask_IR_filter[1] == False
    assert mn.mask_pd_no_filter[1] == True

def test_uplink_responsivity_assignment(mock_node_builder_master, monkeypatch):
    """Verify the spectral responsivity for IR communication."""
    
    def mock_get_resp(name):
        mapping = {
            "IR2PDwF": 0.6,   # IR signal through IR-pass filter
            "IR2PD": 0.85,    # IR signal without filter
        }
        return mapping.get(name, 0.0)

    monkeypatch.setattr(SpectralPhysics, "get_responsivity_by_name", mock_get_resp)
    
    mn = MNManager(mock_node_builder_master)
    
    # Master 0 (With Filter): c_d = 0.6
    assert mn.c_d[0] == 0.6
    # Master 1 (No Filter): c_d = 0.85
    assert mn.c_d[1] == 0.85
    
    # c_d_n (noise) logic should match signal for IR-only context here
    assert mn.c_d_n[0] == 0.6

...                                                                                          [100%]
3 passed in 0.02s


In [5]:
%%ipytest

@pytest.fixture
def mock_ambient_node_builder():
    """Mocks a NodeBuilder for 4 ceiling lamps."""
    class MockNB:
        def __init__(self):
            # 4 lamps in a square pattern on the ceiling (Z=3)
            self.positions = np.array([
                [1.5, 1.5, 3.0], [3.5, 1.5, 3.0],
                [1.5, 2.5, 3.0], [3.5, 2.5, 3.0]
            ])
            self.no_ambient = 4
            self.nT = np.array([[0, 0, -1]] * 4) # All pointing down
            self.m = 1                           # Standard Lambertian
            self.tx_power = 5.0                  # 5 Watts per lamp
            
    return MockNB()

def test_an_manager_initialization(mock_ambient_node_builder):
    """Verify that OTx elements are created with correct geometry and power."""
    an = ANManager(mock_ambient_node_builder)
    
    # Check count
    assert an.OTx_elements.N == 4
    
    # Check that positions were passed correctly
    assert np.array_equal(an.OTx_elements.r[0], [1.5, 1.5, 3.0])
    
    # Check that normals are pointing down (negative Z)
    assert np.all(an.OTx_elements.n[:, 2] == -1.0)
    
    # Check power broadcasting
    # Assuming OpticalTxElements stores power p as (N, 1)
    assert np.all(an.OTx_elements.p == 5.0)

def test_an_manager_empty_builder():
    """Verify behavior with a single node."""
    class SingleNB:
        def __init__(self):
            self.positions = np.array([[2.5, 2.0, 3.0]])
            self.nT = np.array([[0, 0, -1]])
            self.m = 1
            self.tx_power = 10.0
            
    an = ANManager(SingleNB())
    assert an.OTx_elements.N == 1
    assert an.OTx_elements.p[0] == 10.0

..                                                                                           [100%]
2 passed in 0.01s


In [6]:
%%ipytest

@pytest.fixture
def mock_phy_setup():
    """Builds a full environment for physical layer testing."""
    # Mocking Room, SNManager, MNManager, ANManager
    # Assume Room, Surface, and NodeBuilder are defined in previous cells
    
    # 1. Simple Room (2x2x2)
    cfg = {'environment': {'dimensions': [2, 2, 2], 'wall_resolution': (2, 2), 
                           'reflectivity': {'floor': 0.5, 'ceiling': 0.5, 'walls': 0.5}}}
    rb = RoomBuilder(cfg)
    room = Room(rb)
    
    # 2. Master (1 Node)
    class MockMN:
        def __init__(self):
            self.no_masters = 1
            self.OTx_elements = OpticalTxElements(r=[1,1,2], n=[0,0,-1], p=10.0)
            self.ORx_elements = OpticalRxElements(r=[1,1,2], n=[0,0,-1], A=1e-4)
            self.c_d = np.array([0.5])   # IR Uplink Responsivity
            self.c_d_n = np.array([0.1]) # Noise Responsivity
            self.tia = None
            self.BW = 1e6
            self.sensitivity = -90
    
    # 3. Sensor (1 Node)
    class MockSN:
        def __init__(self):
            self.no_sensors = 1
            self.ORx_elements = OpticalRxElements(r=[1,1,0], n=[0,0,1], A=1e-4)
            self.OTx_elements = OpticalTxElements(r=[1,1,0], n=[0,0,1], p=0.5)
            self.c_d = np.array([0.6])   # VLC Downlink Responsivity
            self.c_d_n = np.array([0.05])# Solar Noise Responsivity
            self.ir_flag = 1
            self.rf_flag = 0
            self.tia = None
    
    # 4. Ambient (No artificial lamps for now)
    class MockAN:
        def __init__(self):
            self.OTx_elements = None
            
    return room, MockMN(), MockSN(), MockAN()

def test_downlink_signal_conversion(mock_phy_setup):
    """Verify H -> Prx -> Isig flow for Downlink."""
    room, mn, sn, an = mock_phy_setup
    phy = oPhyGains(room, mn, sn, an)
    phy.compute_downlink()
    
    # Logic: 
    # Prx = H * Ptx (10.0 Watts)
    # Isig = Prx * Responsivity (0.6)
    assert phy.p_d_los.shape == (1, 1)
    assert phy.i_d_los == pytest.approx(phy.p_d_los * 0.6)
    assert hasattr(phy, 'i_d_signal')

def test_uplink_integration(mock_phy_setup):
    """Verify that Uplink is only computed if ir_flag > 0."""
    room, mn, sn, an = mock_phy_setup
    phy = oPhyGains(room, mn, sn, an)
    phy.compute_uplink()
    
    # Sensors transmit 0.5 Watts Uplink
    assert phy.p_u_los.shape == (1, 1)
    # Master converts with 0.5 responsivity
    assert phy.i_u_los == pytest.approx(phy.p_u_los * 0.5)

def test_solar_noise_precursors(mock_phy_setup):
    """Verify that window elements contribute to noise current."""
    room, mn, sn, an = mock_phy_setup
    
    # Add a window to the room manually
    win = Surface(center=[1, 2, 1], dims=(1, 1), const_axis=1, resolution=(2,2), type='window')
    room.add_surface(win)
    
    phy = oPhyGains(room, mn, sn, an)
    phy.compute_ambient()
    
    # Check if solar noise was calculated at sensors
    assert hasattr(phy, 'is_d_noise')
    assert phy.is_d_noise.shape == (1, 1)
    # Solar noise current should use c_d_n (0.05)
    # is_d_los_sum should be (N_window_tiles, N_sensors) summed over axis 0
    assert phy.is_d_noise[0, 0] > 0

...                                                                                          [100%]
3 passed in 0.49s
